# ДЗ-1. Ваш участок ПРАЙМ

Вы — аналитик команды подписки ПРАЙМ. У каждого аналитика свой **участок**: одна таблица, один сегмент и одно окно времени. Это тот самый персональный срез из `prime.assignments`, с которым вы работали на первом семинаре, и больше он ни у кого не повторяется.

В понедельник руководитель просит сводку по участку — восемь задач. В каждой две части: **расчёт** (функция на Python) и **вопрос**, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N». Числа у всех разные, поэтому и выводы у каждого свои.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение к базе и выгрузка вашего участка | вписываете пять значений в ячейку 1.1 и запускаете ячейки по порядку |
| **2. Восемь задач** | условия, заготовки функций, вопросы | пишете код функций и ответы словами |
| **3. Итог** | готовый код: сводка ваших ответов и проверка их формата | запускаете перед сдачей |
| **4. Исследование** | две открытые задачи для тех, кто хочет 9 или 10 | по желанию |

**Оценка.** Части 1–3 — это оценка до 8 баллов: сделали всё аккуратно, выводы верные — 8, и это «отлично». 9 и 10 — только за исследование в части 4. Все критерии — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-1-окружение-и-репозиторий), там же — как сдавать.

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей — **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в **вашем** репозитории как `notebooks/hw1.ipynb`, а JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно — он готов. Единственная ваша правка — пять значений в ячейке 1.1. Номер ячейки — это номер заголовка над ней: «ячейка 1.1» — код сразу под заголовком «1.1 Ваш участок».

Код длинный, потому что аккуратно обходит несколько ловушек, до которых мы дойдём позже. Прочитать его полезно, разбирать каждую строку не обязательно.

| Ячейка | Что делает | Что делаете вы |
|---|---|---|
| 1.1 Ваш участок | хранит пять значений участка | вписываете значения |
| 1.2 Подключение | подключается к базе через ваш `.env` | запускаете |
| 1.3 Строка участка | показывает вашу строку из `prime.assignments` | запускаете, копируете значения в 1.1 и запускаете 1.1 ещё раз |
| 1.4 Запрос | описывает, как достать участок из любой из четырёх таблиц | запускаете |
| 1.5 Выгрузка | достаёт участок в переменную `rows` | запускаете — дальше вы работаете с `rows` |
| 1.6 Формат ответов | служебная функция для итога | запускаете |

### 1.1 Ваш участок

Впишите значения из своей строки `prime.assignments` — строками, в кавычках. Какие они, покажет ячейка 1.3: запустите 1.2 и 1.3, скопируйте напечатанные строки сюда и **запустите эту ячейку ещё раз**, иначе Python продолжит видеть старые значения.

Это **единственное место**, где указан участок. При проверке преподаватель подставит сюда другой участок — и все ответы должны пересчитаться сами. Поэтому ячейку не удаляйте и не пересоздавайте, а значения больше нигде не повторяйте.

In [1]:
TABLE = 'payments'
SEGMENT_COLUMN = 'payment_method'
SEGMENT = 'wallet'
PERIOD_START = '2025-07-01'
PERIOD_END = '2025-09-30'

### 1.2 Подключение

Подключаемся к учебной базе и заводим функцию `query(sql)`: она отправляет SQL-запрос и возвращает результат списком словарей — одна строка результата, один словарь, как на семинарах. Пароля в ноутбуке нет: его знает только ваш `.env`.

Если здесь ошибка `Пакет prime не найден` — ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN` — нет файла `.env` в корне `prime-monitor`.

Соединение открывается на каждый запрос и сразу закрывается: так после сна ноутбука или смены Wi-Fi ничего не отваливается, а общий сервер не держит лишних подключений.

In [2]:
import sys
from collections import Counter, defaultdict  # понадобятся в задачах
from datetime import date
from decimal import Decimal

from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            # Моменты времени в базе хранятся с часовым поясом. День события считаем
            # по UTC — так он у всех одинаковый, какие бы настройки ни стояли у вас.
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        # Соединение не держим: сервер у всей группы общий, а после сна ноутбука
        # старое соединение всё равно мёртвое.
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

подключились к базе как: yuabelaya


### 1.3 Строка участка

Показывает вашу строку из `prime.assignments` — ту же, что на первом семинаре. Чужих строк база не показывает. Скопируйте напечатанное в ячейку 1.1 и запустите 1.1 ещё раз.

За то, что в 1.1 стоят именно ваши значения, отвечаете вы: код с чужим участком спокойно всё посчитает, только не то.

In [3]:
mine = query("select * from prime.assignments where login = current_user")

if mine:
    row = mine[0]
    print("Ваш участок. Скопируйте в ячейку 1.1 и запустите её:\n")
    print(f"TABLE = {row['table_name']!r}")
    print(f"SEGMENT_COLUMN = {row['segment_column']!r}")
    print(f"SEGMENT = {row['segment']!r}")
    print(f"PERIOD_START = {str(row['period_start'])!r}")
    print(f"PERIOD_END = {str(row['period_end'])!r}")
else:
    # Так бывает, когда ноутбук запускает преподаватель под своей ролью.
    print("Строки участка для этого пользователя в базе нет.")

Ваш участок. Скопируйте в ячейку 1.1 и запустите её:

TABLE = 'payments'
SEGMENT_COLUMN = 'payment_method'
SEGMENT = 'wallet'
PERIOD_START = '2025-07-01'
PERIOD_END = '2025-09-30'


### 1.4 Запрос

Четыре таблицы устроены по-разному, но для задач нужно одно и то же. Поэтому запрос приводит любую из них к **пяти полям**:

| Поле | Что это |
|---|---|
| `row_id` | идентификатор строки |
| `actor_id` | участник — тот, чьи это строки |
| `day` | день события, тип `datetime.date` |
| `ok` | состоялось ли событие: `True` или `False` |
| `value` | сколько: деньги, минуты или часы, тип `Decimal` |

Что это значит в **вашей** таблице:

| Таблица | Одна строка — это | Участник | Сегмент | `ok = True` | `ok = False` | `value` |
|---|---|---|---|---|---|---|
| `payments` | попытка списать плату за подписку | подписка | способ оплаты: `card`, `sbp`, `wallet`, `balance` | списание прошло | не прошло или деньги вернули | сумма списания, ₽ |
| `transactions` | покупка клиента в партнёрском сервисе | клиент | канал: `app` — приложение, `web` — сайт, `pos` — касса | покупка прошла | отклонена или проведена и отменена | сумма покупки, ₽; бывает ноль и меньше нуля |
| `service_usage` | сессия в сервисе ПРАЙМ | клиент | устройство | сессия записана корректно: конец позже начала | конец не позже начала | длительность, минуты |
| `support_tickets` | обращение в поддержку | клиент | тема обращения | обращение решено | ещё в работе | время от создания до решения, часы; у нерешённых — `None` |

**Участник** здесь — просто тот, чьи это строки. Не путайте с участниками подписки из `subscription_members`, о которых шла речь на первом семинаре. А у `support_tickets` сумма `value` по дню растёт и от числа обращений, и от медленных решений — это не одно и то же.

Как работает запрос:

1. Блок `acts` выбирает **1000 участников** вашего сегмента за ваше окно. Это выборка, а не весь сегмент: поэтому строк в `rows` меньше, чем вы насчитали в хвосте первого семинара, — там был весь срез. Подгонять под то число не нужно. В ответах словами пишите «в выгрузке» или «среди 1000 подписок / клиентов», а не «по всем картам за квартал». Перемешивание через `md5` делает выбор похожим на случайный, но при каждом запуске — одинаковым.
2. Основной запрос берёт строки этих участников — **только в вашем сегменте и в вашем окне** (тот же фильтр, что в `acts`) — и переименовывает колонки в пять полей.
3. В `{фигурных скобках}` — названия колонок из словаря `COLUMNS`: они подставляются под вашу таблицу. После двоеточия (`:segment`, `:period_start`, `:period_end`) — ваши значения из ячейки 1.1.
4. Граница окна: момент события `>=` первого дня и `<` дня, следующего за последним. Так последний день попадает в окно целиком.

In [4]:
# Как в каждой из четырёх таблиц называются колонки, из которых получаются пять полей.
COLUMNS = {
    "payments": {
        "segment_column": "payment_method",   # колонка сегмента
        "row_id": "payment_id",                # -> row_id
        "actor": "subscription_id",            # -> actor_id
        "moment": "paid_at",                   # момент события -> day
        "ok": "status = 'success'",            # -> ok
        "value": "amount",                     # -> value
    },
    "transactions": {
        "segment_column": "channel",
        "row_id": "transaction_id",
        "actor": "client_id",
        "moment": "occurred_at",
        "ok": "status = 'success'",
        "value": "amount",
    },
    "service_usage": {
        "segment_column": "device_type",
        "row_id": "usage_id",
        "actor": "client_id",
        "moment": "started_at",
        "ok": "ended_at > started_at",
        "value": "extract(epoch from (ended_at - started_at)) / 60.0",      # секунды -> минуты
    },
    "support_tickets": {
        "segment_column": "category",
        "row_id": "ticket_id",
        "actor": "client_id",
        "moment": "created_at",
        "ok": "resolved_at is not null",
        "value": "extract(epoch from (resolved_at - created_at)) / 3600.0",  # секунды -> часы
    },
}

SLICE_SQL = """
with acts as (
    select actor_id
    from (
        select distinct {actor}::text as actor_id
        from prime.{table}
        where {segment_column} = :segment
          and {moment} >= cast(:period_start as date)
          and {moment} <  cast(:period_end as date) + 1
    ) s
    order by md5(actor_id)
    limit 1000
)
select t.{row_id}::text as row_id,
       t.{actor}::text  as actor_id,
       t.{moment}::date as day,
       {ok}             as ok,
       {value}          as value
from prime.{table} t
join acts a on a.actor_id = t.{actor}::text
where t.{segment_column} = :segment
  and t.{moment} >= cast(:period_start as date)
  and t.{moment} <  cast(:period_end as date) + 1
order by t.{row_id}
"""

print("запрос описан")

запрос описан


### 1.5 Выгрузка

Достаём участок из базы в переменную `rows` — список словарей, по одному на строку. **Все задачи ниже работают с `rows`.**

Сколько в выгрузке строк, участников и дней — ячейка не печатает: это вы посчитаете сами в задаче 1.

In [5]:
# В текст запроса подставляются только известные названия таблиц и колонок —
# поэтому сначала проверяем, что значения в ячейке 1.1 из этого списка.
if TABLE not in COLUMNS:
    raise ValueError(f"TABLE должна быть одной из {list(COLUMNS)}, сейчас {TABLE!r}")
if SEGMENT_COLUMN != COLUMNS[TABLE]["segment_column"]:
    raise ValueError(f"у таблицы {TABLE} колонка сегмента называется {COLUMNS[TABLE]['segment_column']!r}")
try:
    window_days = (date.fromisoformat(PERIOD_END) - date.fromisoformat(PERIOD_START)).days + 1
except (TypeError, ValueError):
    raise ValueError("PERIOD_START и PERIOD_END — строки вида '2025-04-01'") from None

# Собираем запрос под вашу таблицу и выполняем. Обычно несколько секунд;
# если висит дольше минуты — см. «Если что-то не работает» в тексте задания.
slice_sql = SLICE_SQL.format(table=TABLE, **COLUMNS[TABLE])
rows = query(slice_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)

# Защита от ошибок в значениях 1.1: пустая выгрузка или дни за пределами окна.
if not rows:
    raise ValueError("выгрузка пустая — проверьте значения в ячейке 1.1")
if len({row["day"] for row in rows}) > window_days:
    raise ValueError("в выгрузке дней больше, чем в окне — граница периода съехала")

print("Выгрузка готова. Так выглядят первые три строки:")
rows[:3]

Выгрузка готова. Так выглядят первые три строки:


[{'row_id': '004aa22e-32e6-428a-886a-d60cc9e1dede',
  'actor_id': 'b7d72947-490c-42f1-97fb-2272e5bd2961',
  'day': datetime.date(2025, 9, 12),
  'ok': True,
  'value': Decimal('199.00')},
 {'row_id': '005d2834-8e4f-4883-9b14-a4c971342314',
  'actor_id': '90799cc4-ae88-447d-8971-c9c398358b8f',
  'day': datetime.date(2025, 9, 29),
  'ok': True,
  'value': Decimal('399.00')},
 {'row_id': '00810eb9-e5b1-4d62-915c-1c0480af509a',
  'actor_id': '5e71e5ee-220e-4a79-b55d-168f13c1a9b3',
  'day': datetime.date(2025, 9, 5),
  'ok': False,
  'value': Decimal('199.00')}]

### 1.6 Формат ответов

Служебная функция: по ней итог в части 3 проверяет, что ответ записан в нужном формате — список, число, дата строкой. **Верно ли посчитано, она не знает и не сообщает** — это проверяет преподаватель. Код ячейки свёрнут — так задумано, просто запустите её.

In [6]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "list из 4 элементов: [int, int, float до 2 знаков, int]",
    2: "int",
    3: "dict: ключи из Mon…Sun, значения float до 2 знаков",
    4: "list [str, int]",
    5: "list из 3 элементов, каждый — list ['ГГГГ-ММ-ДД', float до 2 знаков]",
    6: "list [int, 'ГГГГ-ММ-ДД']",
    7: "list [float до 2 знаков, int]",
    8: "float до 2 знаков",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:  # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def iso_day(x) -> bool:  # дата строкой 'ГГГГ-ММ-ДД'
        try:
            return type(x) is str and date.fromisoformat(x).isoformat() == x
        except ValueError:
            return False

    if answer is None:
        return "не решена: функция вернула None — в ней остался ... или забыт return"
    checks = {
        1: lambda a: type(a) is list and len(a) == 4 and type(a[0]) is int and type(a[1]) is int
                     and money(a[2]) and type(a[3]) is int,
        2: lambda a: type(a) is int,
        3: lambda a: type(a) is dict and set(a) <= {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"}
                     and all(money(v) for v in a.values()),
        4: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is str and type(a[1]) is int,
        5: lambda a: type(a) is list and len(a) == 3
                     and all(type(p) is list and len(p) == 2 and iso_day(p[0]) and money(p[1]) for p in a),
        6: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is int and iso_day(a[1]),
        7: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and type(a[1]) is int,
        8: lambda a: money(a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r}"


print("проверка формата готова")

проверка формата готова


## Часть 2. Восемь задач

У каждой задачи два шага:

1. **Посчитать.** Допишите функцию `taskN(rows)` — она должна вернуть ответ ровно в указанном формате. Имена функций и переменных `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом участке.
2. **Ответить на вопрос.** Для ответа почти всегда нужно досчитать одно-два числа — для этого под вопросом есть пустая ячейка. Сам ответ пишется в ячейке «✍️ Ответ на вопрос N»: дважды щёлкните по ней, **заголовок не трогайте**, замените строку курсивом своим текстом и нажмите `Shift + Enter`.

**Ячейки расчётов тоже выполняются на контрольном участке.** Считайте в них от `rows` и `answer_N`; конкретные даты, `actor_id` и сегмент руками не вписывайте — на другом участке их нет, и ячейка упадёт.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All, и ноутбук не проходит целиком — это минус баллы за воспроизводимость. Верните в тело функции `...`: она вернёт `None`, итог покажет «❌ не решена», а остальное проверится как обычно.

**Про ответы словами.** Код можно писать вместе с ИИ-ассистентом — это нормально, отметьте это в декларации. А ответы пишите сами: здесь оценивается, как думаете вы. **Текст, написанный LLM, получает 0 баллов.** Хороший ответ — 2–4 предложения, в них ваши числа и ваш вывод. Сомнительный вывод баллов не приносит, даже если числа верные.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 3 проверяет только формат. Правильность проверяет преподаватель — на вашем участке и на контрольном.

### Три особенности данных, на которых спотыкаются чаще всего

* **`value` приходит типом `Decimal`, а не `float`.** Это точный тип для денег. `Decimal` складывается с `Decimal` и с целыми числами, а с `float` — нет: `Decimal("1.5") + 0.5` падает с `TypeError`. В ноутбуке С2-идиомы суммы копились от `total = 0.0` — здесь так не выйдет. Копите суммы в `Decimal` (`defaultdict(Decimal)`, `sum(...)` без стартового `0.0`), а к `float` приводите в самом конце: `float(round(сумма, 2))`.
* **В `support_tickets` у нерешённых обращений `value` равно `None`** — времени до решения ещё нет. Проверяйте `row["value"] is not None` там, где складываете `value`, а не `if row["value"]`: второе молча выкинет и честный ноль. Сами строки с `None` из `rows` не удаляйте — это нерешённые обращения, и они нужны в задачах 1, 2 и в вопросе 1.
* **Строки с `ok = True` и отрицательным или нулевым `value` не выбрасывайте.** В `transactions` такие есть — например, покупка на ноль рублей. Считайте их как есть.

Нужные приёмы разобраны в [ноутбуке С2-идиомы](https://github.com/tikhomirovd/python-for-ba-hse-2026/blob/master/02-среда-git-python/семинар/С2-идиомы.ipynb): генераторные выражения, множества, `defaultdict` и `Counter`, `sorted` с ключом, `enumerate`, `try/except`.

### Задача 1. Паспорт участка

Руководитель начинает каждую неделю с одного вопроса: «Сколько у нас всего — и сколько из этого настоящего?» Прежде чем что-то анализировать, надо знать масштаб участка.

**Что посчитать**

Четыре числа по вашей выгрузке `rows`:

1. сколько всего строк;
2. сколько строк с `ok = True` — событий, которые состоялись;
3. сколько `value` набежало по строкам с `ok = True` — выручка, минуты или часы, смотря какая у вас таблица;
4. сколько **различных** дней встречается в выгрузке — по всем строкам, а не только по состоявшимся.

**Формат ответа:** `[всего, с_ok, сумма, дней]` — например `[2222, 2100, 123456.7, 17]`: целое, целое, `float` до двух знаков, целое.

**Подсказка:** `len(rows)`; `sum(1 for row in rows if ...)`; множество дней `{row["day"] for row in rows}`.

In [7]:
def task1(rows: list[dict]) -> list:
    total = len(rows)
    ok_rows = [row for row in rows if row["ok"]]
    ok_count = len(ok_rows)
    total_value = sum((row["value"] for row in ok_rows if row["value"] is not None), Decimal(0))
    days = len({row["day"] for row in rows})
    return [total, ok_count, float(round(total_value, 2)), days]


answer_1 = task1(rows)
answer_1

[1113, 1049, 282951.0, 92]

**Вопрос 1.** **Что не случилось.** Какая доля строк вашей выгрузки не состоялась (`ok = False`)? Что это значит в жизни вашей таблицы — что именно не произошло? Сколько `value` приходится на эти строки — или объясните, почему у вашей таблицы это число не имеет смысла или его нельзя посчитать.

In [8]:
# Числа для ответа на вопрос 1 считайте здесь — по rows и answer_1.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
not_ok = [r for r in rows if not r["ok"]]
share_not_ok = len(not_ok) / len(rows) * 100
value_not_ok = sum((r["value"] for r in not_ok if r["value"] is not None), Decimal(0))
none_count = sum(1 for r in not_ok if r["value"] is None)

print(f"Не состоялось {len(not_ok)} из {len(rows)} строк ({share_not_ok:.1f}%).")
print(f"value несостоявшихся: {float(round(value_not_ok, 2))}.")
print(f"Строк с value=None среди несостоявшихся: {none_count}.")

Не состоялось 64 из 1113 строк (5.8%).
value несостоявшихся: 18336.0.
Строк с value=None среди несостоявшихся: 0.


#### ✍️ Ответ на вопрос 1

Не состоялось 64 оплаты из 1113 - это 5,8 %. В таблице `payments` это неудачные списания с кошелька или возвраты: клиент не оплатил подписку, и доступ, скорее всего, не продлился. На эти строки приходится 18 336 рубля - сумма, которую не собрали или вернули за квартал.

### Задача 2. И получилось, и нет

Самые интересные участники — те, у кого в одном окне было и «получилось», и «не получилось»: оплата сначала не прошла, а потом прошла; одну покупку отклонили, другую провели. Руководитель хочет знать, сколько таких.

**Что посчитать**

Сколько **различных** участников (`actor_id`) имеют в выгрузке хотя бы одну строку с `ok = True` **и** хотя бы одну строку с `ok = False`.

**Формат ответа:** целое число, например `123`. Ноль — тоже законный ответ.

**Подсказка:** два множества участников и операция `&` между ними.

In [9]:
def task2(rows: list[dict]) -> int:
    ok_actors = {row["actor_id"] for row in rows if row["ok"]}
    bad_actors = {row["actor_id"] for row in rows if not row["ok"]}
    return len(ok_actors & bad_actors)


answer_2 = task2(rows)
answer_2

16

**Вопрос 2.** **Почему столько.** Сколько в среднем строк приходится на одного участника в вашей выгрузке? Объясните, почему ответ задачи 2 получился именно таким. Если это ноль или единица — это не ошибка: объясните, откуда он берётся. Подумайте, к чему относится сегмент: к отдельной строке (у одного участника строки могут быть с разными значениями) или к участнику целиком (у всех его строк оно одно).

In [10]:
# Числа для ответа на вопрос 2 считайте здесь — по rows и answer_2.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
actors = {r["actor_id"] for r in rows}
avg_rows = len(rows) / len(actors)
print(f"В выгрузке {len(actors)} участников, среднее число строк на участника: {avg_rows:.2f}.")
print(f"answer_2 = {answer_2}.")
print("Сегмент payment_method относится к строке, в выгрузке оставлены только строки, относящиеся к кактегории wallet"
      "но у одной подписки их может быть несколько — часть успешные, часть нет.")

В выгрузке 1000 участников, среднее число строк на участника: 1.11.
answer_2 = 16.
Сегмент payment_method относится к строке, в выгрузке оставлены только строки, относящиеся к кактегории walletно у одной подписки их может быть несколько — часть успешные, часть нет.


#### ✍️ Ответ на вопрос 2

В выгрузке 1000 участников и 1113 строк, то есть 1,11 строки на участника — у большинства подписок ровно одна попытка списания. Оплата сначала не прошла, а потом прошла у 16 подписок из 1000 (1,6 %). Чтобы у одного `subscription_id` были и успех, и провал, нужно, чтобы он попал в wallet-сегмент минимум дважды с разным исходом. Сегмент относится к строке, а не к участнику, но в выгрузку попадают только wallet-строки, поэтому сочетание успеха и провала редкое.

### Задача 3. Главный день недели

Команда решает, в какой день недели ставить дежурство и запускать рассылки. Нужна раскладка участка по дням недели.

**Что посчитать**

Сумма `value` по строкам с `ok = True` — отдельно для каждого дня недели.

**Формат ответа:** словарь `{'Mon': 12345.67, 'Tue': ...}` — только дни недели, которые встретились; суммы `float` до двух знаков. Название дня берите из списка `WEEKDAYS` по номеру `row["day"].weekday()` (понедельник — 0).

**Подсказка:** `defaultdict(Decimal)`. Почему не `day.strftime('%a')`: результат зависит от языковых настроек — стоит где-нибудь вызвать `locale.setlocale`, и вместо `'Tue'` получится `'вт'`, а ответ разойдётся с форматом.

In [11]:
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]  # WEEKDAYS[0] — понедельник


def task3(rows: list[dict]) -> dict:
    sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"] and row["value"] is not None:
            sums[WEEKDAYS[row["day"].weekday()]] += row["value"]
    return {day: float(round(sums[day], 2)) for day in WEEKDAYS if day in sums}


answer_3 = task3(rows)
answer_3

{'Mon': 33470.0,
 'Tue': 48822.0,
 'Wed': 42848.0,
 'Thu': 40250.0,
 'Fri': 36068.0,
 'Sat': 37257.0,
 'Sun': 44236.0}

**Вопрос 3.** **Сумма или среднее.** Сколько **различных дат** вашей выгрузки приходится на каждый день недели? Считайте даты, а не строки. Какой день лидирует по сумме из задачи 3, а какой — по средней сумме на одну такую дату (сумма из задачи 3, делённая на число дат этого дня недели)? Если лидеры разные — какой ответ вы понесли бы руководителю и почему; если одинаковые — объясните, почему так вышло.

In [12]:
# Числа для ответа на вопрос 3 считайте здесь — по rows и answer_3.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
dates_by_day = defaultdict(set)
for r in rows:
    dates_by_day[WEEKDAYS[r["day"].weekday()]].add(r["day"])

for day in WEEKDAYS:
    if day in dates_by_day:
        n_dates = len(dates_by_day[day])
        total = answer_3.get(day, 0.0)
        avg = total / n_dates if n_dates else 0
        print(f"{day}: дат={n_dates}, сумма={total:.2f}, средняя сумма на дату={avg:.2f}")

if answer_3:
    leader_sum = max(answer_3, key=answer_3.get)
    leader_avg = max(
        (d for d in answer_3 if d in dates_by_day and dates_by_day[d]),
        key=lambda d: answer_3[d] / len(dates_by_day[d])
    )
    print(f"Лидер по сумме: {leader_sum}; лидер по средней сумме на дату: {leader_avg}.")

Mon: дат=13, сумма=33470.00, средняя сумма на дату=2574.62
Tue: дат=14, сумма=48822.00, средняя сумма на дату=3487.29
Wed: дат=13, сумма=42848.00, средняя сумма на дату=3296.00
Thu: дат=13, сумма=40250.00, средняя сумма на дату=3096.15
Fri: дат=13, сумма=36068.00, средняя сумма на дату=2774.46
Sat: дат=13, сумма=37257.00, средняя сумма на дату=2865.92
Sun: дат=13, сумма=44236.00, средняя сумма на дату=3402.77
Лидер по сумме: Tue; лидер по средней сумме на дату: Tue.


#### ✍️ Ответ на вопрос 3

Различных дат на каждый день недели по 13, кроме вторника (14). Лидер по сумме и по средней на дату - вторник (48 822 рубля, 3 487 рубля/день). У вторника на одну дату больше, но он держит первое место и по средней, поэтому руководителю я понесла бы именно вторник, так как по объёму и по интенсивности он первый.

### Задача 4. Самый активный участник

Маркетинг хочет наградить самого активного участника участка — того, у кого больше всего состоявшихся событий. Приз уже куплен.

**Что посчитать**

Участник с наибольшим числом строк с `ok = True` и это число. Если таких несколько — тот, чей `actor_id` меньше при сравнении как текст (так сравнивает `min`).

**Формат ответа:** `[actor_id, число строк]` — например `['aaaaaaaa-1111-2222-3333-bbbbbbbbbbbb', 9]`: текст (`str`) и целое.

**Подсказка:** `Counter` по строкам с `ok = True` считает, сколько состоявшихся событий у каждого участника; `max` находит наибольшее число, `min` — наименьший `actor_id` среди тех, у кого оно такое.

In [13]:
def task4(rows: list[dict]) -> list:
    cnt = Counter(row["actor_id"] for row in rows if row["ok"])
    if not cnt:
        return ["", 0]
    best = max(cnt.values())
    winners = [actor for actor, c in cnt.items() if c == best]
    return [min(winners), best]


answer_4 = task4(rows)
answer_4

['80548efc-13fb-42bf-b4ed-0cdf5a9af9d0', 3]

**Вопрос 4.** **Кто победил на самом деле.** Сколько участников делят первое место? Если больше одного — что на самом деле выбрало вашего «победителя»: данные или правило? И что в вашей таблице вообще значит «самый активный» — стоит ли маркетингу вручать приз по такому рейтингу?

In [14]:
# Числа для ответа на вопрос 4 считайте здесь — по rows и answer_4.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
cnt = Counter(r["actor_id"] for r in rows if r["ok"])
best = max(cnt.values()) if cnt else 0
winners = [a for a, c in cnt.items() if c == best]

print(f"Максимум состоявшихся событий: {best}; участников с таким числом: {len(winners)}.")

Максимум состоявшихся событий: 3; участников с таким числом: 1.


#### ✍️ Ответ на вопрос 4

Максимум 3 успешных списания, и он только у одного участника, поэтому правило min(actor_id) не сработало: победителя выбрали данные. Но в payments «самый активный» - это подписка, которую чаще всего успешно списывали, это скорее про число попыток, чем про ценность клиента, поэтому вручать приз по такому клиенту я бы не стала без дополнительных метрик, например, MRR (Monthly Recurring Revenue) или LTV клиента.

### Задача 5. Три пиковых дня

Чтобы планировать нагрузку на команду и серверы, нужно знать пиковые дни окна.

**Что посчитать**

Три даты с наибольшей суммой `value` по строкам с `ok = True` — по убыванию суммы. Если суммы равны, раньше идёт более ранняя дата.

**Формат ответа:** `[['2024-03-15', 98765.43], ['2024-03-02', 91200.1], ['2024-03-21', 90000.0]]` — дата строкой `'ГГГГ-ММ-ДД'`, сумма `float` до двух знаков.

**Подсказка:** сначала словарь «день → сумма», потом `sorted(..., key=...)`: ключ-кортеж `(-сумма, день)` сортирует по убыванию суммы, а при равенстве — по возрастанию даты. Дата строкой — `day.isoformat()`.

In [15]:
def task5(rows: list[dict]) -> list:
    sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"] and row["value"] is not None:
            sums[row["day"]] += row["value"]
    top = sorted(sums.items(), key=lambda kv: (-kv[1], kv[0]))[:3]
    return [[day.isoformat(), float(round(total, 2))] for day, total in top]


answer_5 = task5(rows)
answer_5

[['2025-09-24', 6579.0], ['2025-09-30', 5382.0], ['2025-09-17', 4981.0]]

**Вопрос 5.** **Откуда пик.** Какие это дни недели, сколько среди них суббот и воскресений? Сформулируйте одну гипотезу, почему пик пришёлся на эти даты, и напишите, какой расчёт на данных её подтвердил бы или опроверг.

In [16]:
# Числа для ответа на вопрос 5 считайте здесь — по rows и answer_5.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
for d, s in answer_5:
    dt = date.fromisoformat(d)
    print(f"{d} — {WEEKDAYS[dt.weekday()]}, сумма {s:.2f}")
weekends = sum(1 for d, _ in answer_5 if date.fromisoformat(d).weekday() >= 5)
print(f"Выходных среди топ-3: {weekends}.")

2025-09-24 — Wed, сумма 6579.00
2025-09-30 — Tue, сумма 5382.00
2025-09-17 — Wed, сумма 4981.00
Выходных среди топ-3: 0.


#### ✍️ Ответ на вопрос 5

Топ-3 дня: 
* 2025-09-24 (ср),
* 2025-09-30 (вт),
* 2025-09-17 (ср)

Выходных среди них нет. 

Гипотеза: пики привязаны к массовым датам списаний в середине и конце месяца. Проверить её можно так: посмотреть распределение списаний по дню месяца, если много подписок привязано к 17-му и 24-му числам, гипотеза подтвердится, если пики «размазаны», то дело не в датах.

### Задача 6. Когда набралась половина

Финансам важно, равномерно ли набирается сумма внутри окна: к какому дню набегает половина суммы по состоявшимся событиям.

**Что посчитать**

1. Возьмите дни, в которых есть хотя бы одна строка с `ok = True`, и расставьте по возрастанию даты.
2. Идите по ним и копите сумму `value` по строкам с `ok = True`.
3. Найдите первый день, на котором накопленное **достигло половины** суммы из пункта 3 задачи 1 (всё `value` по строкам с `ok = True`) или превысило её, и его порядковый номер в списке дней из шага 1, считая с 1.

**Формат ответа:** `[номер, 'ГГГГ-ММ-ДД']` — например `[12, '2024-03-15']`.

**Подсказка:** `sums` — словарь «день → сумма», как в задаче 5. `for number, day in enumerate(sorted(sums), start=1):` — `enumerate` сам считает номер.

In [17]:
def task6(rows: list[dict]) -> list:
    sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"] and row["value"] is not None:
            sums[row["day"]] += row["value"]
    days_sorted = sorted(sums)
    total = sum(sums.values(), Decimal(0))
    half = total / 2
    cum = Decimal(0)
    for number, day in enumerate(days_sorted, start=1):
        cum += sums[day]
        if cum >= half:
            return [number, day.isoformat()]
    if days_sorted:
        return [len(days_sorted), days_sorted[-1].isoformat()]
    return [0, ""]


answer_6 = task6(rows)
answer_6

[49, '2025-08-18']

**Вопрос 6.** **Равномерно ли.** Какую долю дней заняла первая половина суммы: номер дня из задачи 6, делённый на число дней в списке из шага 1? Номер дня целый, поэтому при равномерном накоплении получилось бы около половины, округлённой вверх (15 из 30, 16 из 31) — сравнивайте с этим, а не с ровными 50 %. У вас раньше или позже — и что это говорит о том, как менялся участок внутри окна? Если вышло у середины — так и напишите.

In [18]:
# Числа для ответа на вопрос 6 считайте здесь — по rows и answer_6.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
sums = defaultdict(Decimal)
for r in rows:
    if r["ok"] and r["value"] is not None:
        sums[r["day"]] += r["value"]
days_sorted = sorted(sums)
share_days = answer_6[0] / len(days_sorted) if days_sorted else 0

print(f"Дней с ok=True: {len(days_sorted)}.")
print(f"Половина набралась к дню {answer_6[0]} ({answer_6[1]}).")
print(f"Это {share_days:.1%} списка дней. Для равномерного накопления было бы около 50%.")

Дней с ok=True: 92.
Половина набралась к дню 49 (2025-08-18).
Это 53.3% списка дней. Для равномерного накопления было бы около 50%.


#### ✍️ Ответ на вопрос 6

Половина суммы набралась к 49-му дню из 92 - это 53,3 %, чуть больше середины. Значит, первая половина срока дала меньше половины выручки, а вторая больше: к концу квартала выручка стала стремительно увеличиваться. Это не плохо и не хорошо само по себе, но говорит, что сравнивать месяцы между собой по этой тысяче без поправки на сезон нельзя.

### Задача 7. Функция, которая не падает

Соседний отдел присылает суммы строками — так, как их набрал оператор: с запятой, с пробелами, иногда с прочерком. Нужна функция, которая превращает такую строку в число и не роняет ежедневный отчёт. Данные в этой задаче у всех одинаковые — список `RAW`.

**Что посчитать**

1. Функция `to_value(x)`: получает строку (или `None`) и возвращает `float` — или `None`, если число разобрать нельзя. Падать нельзя ни на одном значении. Правила записи:
   * точка и запятая — десятичный разделитель: `'89.10'` → `89.1`, `'349,90'` → `349.9`;
   * пробел и неразрывный пробел внутри числа — разделители тысяч: `'1 499,50'` → `1499.5`;
   * пробелы по краям ничего не значат;
   * прочерк `'—'` и пустая строка — не число: `None`, а не `0.0`.
2. По списку `RAW`: сумма всего, что разобралось, и сколько значений отброшено. `None` тоже считается отброшенным.

**Формат ответа:** `[сумма, отброшено]` — `float` до двух знаков и целое.

**Подсказка:** `try: ... except ValueError: ...` — ловите именно `ValueError`, а не всё подряд. `None` проверьте до `try`: `float(None)` бросает `TypeError`, а не `ValueError`. `x.split()` делит строку по любым пробелам, включая неразрывный.

In [19]:
RAW = ["199.00", "1 499,50", None, "—", "349,90", "2\u00a0100", "", "  89.10  "]

assert RAW[1][1] == " " and RAW[5][1] == "\u00a0"


def to_value(x: str | None) -> float | None:
    if x is None or not isinstance(x, str):
        return None
    s = x.strip()
    if s == "" or s == "—":
        return None

    s = s.replace(" ", "").replace("\u00a0", "")

    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    else:
        s = s.replace(",", ".")

    try:
        return float(s)
    except ValueError:
        return None


def task7(raw: list) -> list:
    total = 0.0
    dropped = 0
    for x in raw:
        val = to_value(x)
        if val is None:
            dropped += 1
        else:
            total += val
    return [round(total, 2), dropped]


answer_7 = task7(RAW)
answer_7

[4237.5, 3]

**Вопрос 7.** **Что опаснее.** Представьте ежедневный отчёт о выручке, построенный на этой функции. Что опаснее: функция, которая падает на `'—'`, или функция, которая молча возвращает `None`? Что бы вы добавили в отчёт, чтобы потеря была видна руководителю? Опирайтесь на свой ответ задачи 7: сколько из восьми значений было бы потеряно.

In [20]:
# Числа для ответа на вопрос 7 считайте здесь — по rows и answer_7.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
print(f"answer_7 = {answer_7}.")
print(f"Потеряно {answer_7[1]} из {len(RAW)} значений ({answer_7[1]/len(RAW):.1%}).")

answer_7 = [4237.5, 3].
Потеряно 3 из 8 значений (37.5%).


#### ✍️ Ответ на вопрос 7

Потеряно 3 из 8 значений (37,5 %). Опаснее `None`, потому что падение на `'—'` сразу видно и заставит починить парсер, а потеря трети выручки из-за `None` в ежедневном отчёте пройдёт незаметно и исказит цифры вниз. В отчёт стоит добавить число и долю отброшенных значений и потерянную сумму, тогда потеря станет видимой и её можно будет эскалировать.

### Задача 8. Поправка на день недели

Аналитики из соседней команды прислали коэффициенты дней недели, чтобы сравнивать недели между собой. Прислали только будни — как это обычно и бывает.

**Что посчитать**

1. Для каждого дня недели возьмите сумму `value` по строкам с `ok = True` — как в задаче 3, но **до округления**.
2. Приведите её к `float` и умножьте на коэффициент дня из `WEIGHTS`. Если дня в справочнике нет — коэффициент `1.0`.
3. Сложите все произведения. Округлите до двух знаков **только итог**.

**Формат ответа:** одно число `float` до двух знаков, например `123456.78`.

**Подсказка:** `WEIGHTS.get(day, 1.0)` возвращает коэффициент или `1.0`, если дня нет. `Decimal * float` падает с `TypeError` — сначала `float(...)`.

In [21]:
WEIGHTS = {"Mon": 1.00, "Tue": 1.05, "Wed": 1.05, "Thu": 1.00, "Fri": 0.95}


def task8(rows: list[dict]) -> float:
    sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"] and row["value"] is not None:
            sums[WEEKDAYS[row["day"].weekday()]] += row["value"]

    weighted = 0.0
    for day, val in sums.items():
        weighted += float(val) * WEIGHTS.get(day, 1.0)
    return round(weighted, 2)


answer_8 = task8(rows)
answer_8

285731.1

**Вопрос 8.** **Что изменила поправка.** Какие дни недели пошли с коэффициентом по умолчанию? На сколько процентов (с точностью до десятых) взвешенная сумма отличается от обычной суммы из задачи 1 — процент считайте от обычной суммы? В какой ситуации на данных ПРАЙМ такой `.get` с умолчанием спрятал бы ошибку, вместо того чтобы о ней сообщить?

In [22]:
# Числа для ответа на вопрос 8 считайте здесь — по rows и answer_8.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
normal = answer_1[2]
weighted = answer_8
diff = (weighted - normal) / normal * 100 if normal else 0
default_days = [d for d in WEEKDAYS if d in answer_3 and d not in WEIGHTS]

print(f"Обычная сумма: {normal:.2f}; взвешенная: {weighted:.2f}; отличие: {diff:+.1f}%.")
print(f"Коэффициент 1.0 применился к: {', '.join(default_days) if default_days else 'нет'}.")

Обычная сумма: 282951.00; взвешенная: 285731.10; отличие: +1.0%.
Коэффициент 1.0 применился к: Sat, Sun.


#### ✍️ Ответ на вопрос 8

С коэффициентом по умолчанию вошли суббота и воскресенье - их нет в `WEIGHTS`, потому что прислали только будни. Взвешенная сумма 285 731,10 против обычной 282 951,00 отличается на +1,0 %. Такой `.get` с умолчанием спрячет ошибку, если справочник задумывался на все семь дней, а в нём случайно забыли будний день: код молча подставит 1.0 и не сообщит, что справочник неполный. Лучше явно проверять, что все встреченные дни есть в `WEIGHTS`.

## Часть 3. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода — служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [23]:
import json


def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


# Участок из ячейки 1.1 и ответы answer_1 … answer_8; у каждого ответа проверяем формат.
params = {"table_name": TABLE, "segment_column": SEGMENT_COLUMN, "segment": SEGMENT,
          "period_start": PERIOD_START, "period_end": PERIOD_END}
answers = {n: globals().get(f"answer_{n}") for n in range(1, 9)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n}: {answer!r}")
    print("   ✅ формат в порядке" if formats[n] is None else f"   ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» в части 2.")
print("\n--- служебная строка, не удаляйте ---")
print("HW1_ANSWERS=" + json.dumps(
    {"slice": params, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

задача 1: [1113, 1049, 282951.0, 92]
   ✅ формат в порядке
задача 2: 16
   ✅ формат в порядке
задача 3: {'Mon': 33470.0, 'Tue': 48822.0, 'Wed': 42848.0, 'Thu': 40250.0, 'Fri': 36068.0, 'Sat': 37257.0, 'Sun': 44236.0}
   ✅ формат в порядке
задача 4: ['80548efc-13fb-42bf-b4ed-0cdf5a9af9d0', 3]
   ✅ формат в порядке
задача 5: [['2025-09-24', 6579.0], ['2025-09-30', 5382.0], ['2025-09-17', 4981.0]]
   ✅ формат в порядке
задача 6: [49, '2025-08-18']
   ✅ формат в порядке
задача 7: [4237.5, 3]
   ✅ формат в порядке
задача 8: 285731.1
   ✅ формат в порядке

Не забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» в части 2.

--- служебная строка, не удаляйте ---
HW1_ANSWERS={"slice": {"table_name": "payments", "segment_column": "payment_method", "segment": "wallet", "period_start": "2025-07-01", "period_end": "2025-09-30"}, "answers": {"1": [1113, 1049, 282951.0, 92], "2": 16, "3": {"Mon": 33470.0, "Tue": 48822.0, "Wed": 42848.0, "Thu": 40250.0, "Fri": 36068.0, "Sat": 37257.0, "Sun": 44236.

## Часть 4. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–3 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: два задания, каждое до 1 балла сверху. Засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом в ячейках ниже и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждое исследование оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, названо, чего ваш расчёт **не** доказывает | 0,3 |
| код исследования выполняется у преподавателя целиком — в том числе на контрольном участке | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Код исследования, как и всё остальное, считайте от значений ячейки 1.1: на контрольном участке он должен отработать без правок.

База умеет считать сама: `count(*)`, `count(distinct …)`, `sum(…)`, условие внутри агрегата — `count(*) filter (where …)`, разбивка — `group by`. Весь сегмент в Python не выгружайте: запрос дольше 2 минут сервер оборвёт. Функция `query` и словарь `COLUMNS` из части 1 здесь пригодятся.

### Исследование А. Можно ли верить вашей тысяче?

Руководитель прочитал сводку и спрашивает: «Вы смотрели тысячу участников, а в сегменте их гораздо больше. Насколько ваши числа верны для всего сегмента? Если в следующем месяце value на участника упадёт на 5 %, мы это заметим по такой тысяче?»

**Что сделать**

1. По **всему** вашему сегменту за окно — одним запросом к базе, не выгружая строки — посчитайте: сколько участников, долю строк с `ok = True`, `value` на участника (сумма `value` по строкам с `ok = True`, делённая на число различных участников).
2. Те же показатели — по вашей тысяче, из `rows`.
3. Возьмите **не меньше 20 других тысяч** участников того же сегмента и окна и посчитайте показатели для каждой.
4. Ответьте: насколько ваша тысяча отличается от всего сегмента? Каков разброс между тысячами? Заметно ли по одной тысяче падение `value` на участника на 5 % — и сколько участников нужно брать, чтобы такое падение было заметно?

**Подсказки.** Другая тысяча — это другое перемешивание. Ячейку 1.4 не меняйте — сделайте копию запроса в своей ячейке: `salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(table=TABLE, **COLUMNS[TABLE])` и вызывайте `query(salted_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END, salt=str(i))` с разными `i`. Разброс удобно описать стандартным отклонением (`statistics.pstdev`). Как разброс зависит от размера выборки — поищите «стандартная ошибка среднего».

In [24]:
# Исследование А: ваш код. Всё — от значений ячейки 1.1.
import statistics
FULL_SQL = """
with s as (
    select t.{actor}::text as actor_id,
           count(*) as rows_total,
           count(*) filter (where {ok}) as rows_ok,
           coalesce(sum({value}) filter (where {ok}), 0) as value_ok
    from prime.{table} t
    where t.{segment_column} = :segment
      and t.{moment} >= cast(:period_start as date)
      and t.{moment} <  cast(:period_end as date) + 1
    group by 1
)
select count(*)               as actors,
       sum(rows_total)        as rows_total,
       sum(rows_ok)           as rows_ok,
       sum(value_ok)          as value_ok
from s
""".format(table=TABLE, **COLUMNS[TABLE])

full = query(FULL_SQL, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)[0]
full_actors = full["actors"]
full_ok_share = full["rows_ok"] / full["rows_total"]
full_value_per_actor = float(full["value_ok"]) / full_actors
print("Весь сегмент:")
print(f"участников           = {full_actors}")
print(f"доля ok=True         = {full_ok_share:.4f}")
print(f"value на участника   = {full_value_per_actor:.4f}")

# тысяча из rows 
my_actors = {r["actor_id"] for r in rows}
my_ok_share = sum(1 for r in rows if r["ok"]) / len(rows)
my_value_ok = sum((r["value"] for r in rows if r["ok"] and r["value"] is not None), Decimal(0))
my_value_per_actor = float(my_value_ok) / len(my_actors)
print("\nтысяча:")
print(f"участников           = {len(my_actors)}")
print(f"доля ok=True         = {my_ok_share:.4f}")
print(f"value на участника   = {my_value_per_actor:.4f}")

# Ещё 20 других тысяч
salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(
    table=TABLE, **COLUMNS[TABLE]
)
others = []
for i in range(20):
    ri = query(salted_sql, segment=SEGMENT,
               period_start=PERIOD_START, period_end=PERIOD_END, salt=str(i))
    if not ri:
        continue
    a = {r["actor_id"] for r in ri}
    ok_share = sum(1 for r in ri if r["ok"]) / len(ri)
    val = sum((r["value"] for r in ri if r["ok"] and r["value"] is not None), Decimal(0))
    per_actor = float(val) / len(a)
    others.append((len(a), ok_share, per_actor))

sizes = [x[0] for x in others]
ok_shares = [x[1] for x in others]
per_actors = [x[2] for x in others]

print("\nДругие 20 тысяч:")
print(f"участников: медиана {statistics.median(sizes):.0f}, "
      f"разброс [{min(sizes)}; {max(sizes)}]")
print(f"доля ok=True: среднее {statistics.mean(ok_shares):.4f}, "
      f"станд. откл. {statistics.pstdev(ok_shares):.4f}")
print(f"value на участника: среднее {statistics.mean(per_actors):.2f}, "
      f"станд. откл. {statistics.pstdev(per_actors):.2f}")

# Заметно ли падение 5% 
mean_v = statistics.mean(per_actors)
sd_thousand = statistics.pstdev(per_actors)          # ст. откл. между тысячными средними
sd_individual = sd_thousand * (1000 ** 0.5)          # пересчёт к инд. значениям
threshold_5pct = 0.05 * mean_v

# тысяча vs среднее по 20 другим - на сколько сигм отличается
z_value = (my_value_per_actor - mean_v) / sd_thousand
z_ok    = (my_ok_share       - statistics.mean(ok_shares)) / statistics.pstdev(ok_shares)
print(f"\nНаша тысяча vs среднее по 20: z(value/уч) = {z_value:+.2f}, z(ok) = {z_ok:+.2f}")

# Сколько нужно участников, чтобы 2*SE < 5% от среднего
n_needed = (2 * sd_individual / threshold_5pct) ** 2
print(f"ст. откл. индивидуальных значений = {sd_individual:.2f}")
print(f"Порог 5% от среднего: {threshold_5pct:.2f}")
print(f"2×SE на n=1000: {2 * sd_thousand:.2f}")
print(f"Нужно примерно {int(n_needed) + 1} участников, чтобы 5%-е падение было заметно.")

Весь сегмент:
участников           = 44299
доля ok=True         = 0.9549
value на участника   = 292.3320

тысяча:
участников           = 1000
доля ok=True         = 0.9425
value на участника   = 282.9510

Другие 20 тысяч:
участников: медиана 1000, разброс [1000; 1000]
доля ok=True: среднее 0.9527, станд. откл. 0.0060
value на участника: среднее 292.13, станд. откл. 4.79

Наша тысяча vs среднее по 20: z(value/уч) = -1.92, z(ok) = -1.70
ст. откл. индивидуальных значений = 151.51
Порог 5% от среднего: 14.61
2×SE на n=1000: 9.58
Нужно примерно 431 участников, чтобы 5%-е падение было заметно.


#### ✍️ Исследование А

Как проверяли. Сравнили нашу тысячу с агрегатом по всему сегменту. Разброс между выборками оценили по 20 другим случайным тысячам того же сегмента и окна с помощью перемешивания. По станддартному отклонению между тысячными средними пересчитали сигму индивидуальных значений и нашли размер выборки, при котором 5 %-е падение превышает 2*SE.

Что получилось.

Весь сегмент: 44 299 участников, доля ok = 0,9549, value на участника = 292,33 рубля.

Наша тысяча: 1 000 участников, доля ok = 0,9425, value на участника = 282,95 рублей, что на 9,4 рубля (3,2 %) ниже сегмента и на 1,9 сигм ниже среднего по другим 20 тысячам. По ok тоже чуть ниже (z окло −1,7).

20 других тысяч: ok = 0,9527 +/- 0,0060; value на участника = 292,13 +/- 4,79 рубля.

Сигма индивидуальных значений value на участника примерно равна 151,4 ₽. Порог 5 % равен 14,6 рублей. Нужно примерно 430 участников (получилось 431), чтобы 2*SE < 5 %.

Что это значит и чего расчёт не доказывает. Тысяча достаточна, чтобы заметить падение 5 %: 2*SE при n = 1000 равно примерно 9,6 рублей, что меньше порога 14,6 рублей; минимум - 431 участник. Но наша конкретная тысяча случайно оказалась на 1,9 сигм ниже среднего по двадцати другим, поэтому «в лоб» сравнивать её абсолютное число (283 рубля) с полным сегментом (292 рубля) и делать вывод «всё плохо» нельзя. Здесь будто правильнее сравнивать ту же самую тысячу во времени или с медианой 20 тысяч. Расчёт не доказывает причину падения (например, сезонность, изменение состава подписок) и не учитывает хвосты распределения value внутри участника - крупные плательщики могут сдвинуть среднее, поэтому стоит отдельно смотреть медиану и квантили.

### Исследование Б. Ваш сегмент особенный?

Маркетинг заметил, что в одних сегментах участники приносят заметно больше, чем в других, и хочет перераспределить бюджет в пользу «ценных» сегментов. Руководитель просит проверить: правда ли ваш сегмент отличается от соседних — и чем именно.

**Что сделать**

1. Для **всех сегментов** вашей таблицы за ваше окно посчитайте: число участников, строк на участника, долю строк с `ok = True`, `value` на одну строку с `ok = True` и `value` на участника.
2. Разложите `value` на участника на множители и найдите, какой из них даёт различие между сегментами.
3. Проверьте, держится ли картина в предыдущем окне той же длины. Данные начинаются 6 января 2025 года: если предыдущее окно начинается раньше, оно неполное — тогда сравнивайте доли и значения на участника или на строку, а не число участников и строк.
4. Вывод для маркетинга: «ценный» ли ваш сегмент; что на самом деле различается; какое решение по бюджету из этого следует — и чего ваши данные не доказывают.

**Подсказка.** Здесь нужны не выгрузка `rows`, а агрегаты по всей таблице за окно с разбивкой `group by` колонке сегмента.

In [25]:
from datetime import timedelta

SEG_SQL = """
with s as (
    select t.{segment_column}::text as seg,
           t.{actor}::text          as actor_id,
           count(*)                 as rows_total,
           count(*) filter (where {ok}) as rows_ok,
           coalesce(sum({value}) filter (where {ok}), 0) as value_ok
    from prime.{table} t
    where t.{moment} >= cast(:period_start as date)
      and t.{moment} <  cast(:period_end as date) + 1
    group by 1, 2
)
select seg,
       count(*)         as actors,
       sum(rows_total)  as rows_total,
       sum(rows_ok)     as rows_ok,
       sum(value_ok)    as value_ok
from s
group by seg
order by seg
""".format(table=TABLE, **COLUMNS[TABLE])


def summarize(seg_rows):
    out = []
    for r in seg_rows:
        # sum(count(*)) и sum(value) в Postgres возвращают numeric/Decimal —
        # приводим к int/float до арифметики, иначе float/Decimal бросает TypeError. - гпт подправил этот момент
        a  = int(r["actors"] or 0)
        rt = int(r["rows_total"] or 0)
        ro = int(r["rows_ok"] or 0)
        vo = float(r["value_ok"] or 0)
        out.append({
            "seg": r["seg"],
            "actors": a,
            "rows_per_actor": rt / a if a else 0.0,
            "ok_share":       ro / rt if rt else 0.0,
            "value_per_ok_row": vo / ro if ro else 0.0,
            "value_per_actor":  vo / a  if a else 0.0,
        })
    return out


cur = summarize(query(SEG_SQL, period_start=PERIOD_START, period_end=PERIOD_END))
print("Текущее окно:")
for x in cur:
    print(f"{x['seg']:<8} участников={x['actors']:>6}  "
          f"строк/уч={x['rows_per_actor']:.3f}  ok={x['ok_share']:.3f}  "
          f"руб/ok-строку={x['value_per_ok_row']:.2f}  руб/уч={x['value_per_actor']:.2f}")

sd, ed = date.fromisoformat(PERIOD_START), date.fromisoformat(PERIOD_END)
win = (ed - sd).days + 1
prev_ed = sd - timedelta(days=1)
prev_sd = prev_ed - timedelta(days=win - 1)
print(f"\nПредыдущее окно: {prev_sd} … {prev_ed} (длина {win} дн.)")

if prev_sd < date(2025, 1, 6):
    print("  Окно неполное (данные начинаются 2025-01-06) — сравниваем доли "
          "и значения на участника/строку, а не число участников.")
else:
    prev = summarize(query(SEG_SQL, period_start=prev_sd.isoformat(),
                           period_end=prev_ed.isoformat()))
    print("Предыдущее окно:")
    for x in prev:
        print(f"{x['seg']:<8} участников={x['actors']:>6}  "
              f"строк/уч={x['rows_per_actor']:.3f}  ok={x['ok_share']:.3f}  "
              f"руб/ok-строку={x['value_per_ok_row']:.2f}  руб/уч={x['value_per_actor']:.2f}")

print("\nРазложение руб/уч = (строк/уч) × (ok) × (руб/ok-строку):")
for x in cur:
    print(f"{x['seg']:<8} {x['rows_per_actor']:.3f} × {x['ok_share']:.3f} × "
          f"{x['value_per_ok_row']:.2f} = {x['value_per_actor']:.2f}")

Текущее окно:
balance  участников= 25609  строк/уч=1.062  ok=0.956  руб/ok-строку=273.52  руб/уч=277.66
card     участников=138103  строк/уч=1.700  ok=0.955  руб/ok-строку=275.26  руб/уч=446.98
sbp      участников=102393  строк/уч=1.365  ok=0.954  руб/ok-строку=274.80  руб/уч=357.86
wallet   участников= 44299  строк/уч=1.115  ok=0.955  руб/ok-строку=274.45  руб/уч=292.33

Предыдущее окно: 2025-03-31 … 2025-06-30 (длина 92 дн.)
Предыдущее окно:
balance  участников= 22633  строк/уч=1.059  ok=0.954  руб/ok-строку=274.81  руб/уч=277.87
card     участников=122263  строк/уч=1.691  ok=0.955  руб/ok-строку=274.68  руб/уч=443.55
sbp      участников= 90510  строк/уч=1.361  ok=0.955  руб/ok-строку=275.09  руб/уч=357.33
wallet   участников= 39160  строк/уч=1.112  ok=0.955  руб/ok-строку=275.27  руб/уч=292.48

Разложение руб/уч = (строк/уч) × (ok) × (руб/ok-строку):
balance  1.062 × 0.956 × 273.52 = 277.66
card     1.700 × 0.955 × 275.26 = 446.98
sbp      1.365 × 0.954 × 274.80 = 357.86
wallet   1.

#### ✍️ Исследование Б

Как проверяли. По таблице payments за окно с 1 июля 2025 года по 30 сентября 2025 года посчитали по каждому сегменту (card, sbp, wallet, balance) число подписок, строк на подписку, долю успешных списаний, кол-во рублей на успешную строку и рублей на подписку. Разложили рубли/подписку в произведение строк/уч * ok * руб/ok-строку. Ту же разбивку получили за предыдущее окно такой же длины с 31 марта 2025 года по 30 июня 2025 года.

Что получилось.

Доля ok = 0,955 и руб на успешную строку = 274 ₽ почти совпадают у всех четырёх сегментов и в текущем, и в предыдущем окне.

Кол-во рублей на подписку различается в 1,6 раза: card 447 руб > sbp 358 руб > wallet 292 руб > balance 278 руб.

Всё различие объясняется множителем строк на участника: card 1,70 > sbp 1,37 > wallet 1,12 > balance 1,06.

В предыдущем окне картина та же: card 443,6 руб, sbp 357,3 руб, wallet 292,5 руб, balance 277,9 руб, при тех же разложениях. Значит, наблюдение устойчивое, а не разовый выброс.

Что это значит и чего расчёт не доказывает. Наш сегмент wallet не ценнее и не дешевле по сути, так как по количеству денег на успешное списание он такой же, как остальные, и доля успеха у всех одинаковая. Разница в деньгах на подписку возникает только из-за частоты попыток списания: с карты их в среднем 1,7 за квартал, с кошелька - 1,1. Перераспределять маркетинговый бюджет в пользу самого дорогого сегмента card по одному показателю руб/подписку не стоит, так можно профинансировать более частое списание, а не более выгодных клиентов. Правильнее сначала разобраться, откуда берётся эта разница: автосписание с карты (в том числе повторные попытки после неудач) против ручного пополнения кошелька. Расчёт не доказывает причинно-следственную связь между способом оплаты и выручкой, не учитывает LTV, удержание и возвраты и опирается только на два окна по 92 дня, следовательно, для решения по бюджету этого мало, нужен либо A/B тест, либо анализ в динамике нескольких окон с учётом состава клиентов.

## Напоследок

Что вас удивило в вашем участке? Одна-две фразы в ячейке ниже — по желанию и без баллов.

И пройдите, пожалуйста, **[короткий опрос о том, как вам это задание](https://forms.gle/5tNT4P4diHVyzhdFA)** — пара минут. Следующие задания соберу с учётом ответов.

#### ✍️ Что удивило

Интересно, что пики пришлись на середину и конец сентября, а не на выходные, но вцелом, если рассуждать в контексте подписки, то может быть это и логично